# 08 Advanced Challenge - Data QC and Outlier Visualization in R

## Biochemistry question

In this synthetic assay dataset, which measurements should be reviewed before interpreting the overall pattern?


In [ ]:
library(tidyverse)

df <- read_csv("../data/qc_outlier_sample.csv")

df_qc <- df %>%
  group_by(drug_name, concentration_uM) %>%
  mutate(
    group_mean = mean(cell_viability_percent),
    group_sd = sd(cell_viability_percent),
    z_score = (cell_viability_percent - group_mean) / group_sd,
    qc_flag = if_else(abs(z_score) > 1.5, "review", "ok")
  ) %>%
  ungroup()

df_qc

In [ ]:
df_qc %>%
  mutate(plot_concentration = if_else(concentration_uM == 0, 0.001, concentration_uM)) %>%
  ggplot(aes(x = plot_concentration, y = cell_viability_percent, color = qc_flag, shape = drug_name)) +
  geom_point(size = 3) +
  scale_x_log10() +
  labs(
    title = "QC Scatter Plot: Possible Outliers",
    x = "Concentration (uM, log scale; 0 plotted as 0.001)",
    y = "Cell Viability (%)"
  )

In [ ]:
ggplot(df_qc, aes(x = drug_name, y = cell_viability_percent, color = drug_name)) +
  geom_boxplot(outlier.shape = NA) +
  geom_jitter(width = 0.1, size = 2) +
  labs(title = "Cell Viability Distribution by Drug")

## Interpretation Questions

1. Which points were flagged for review?
2. Are flagged points always wrong?
3. Which language made the QC logic easier to read?
4. How could this QC view support a future educational BioDose workflow?

## Limitations

- This is synthetic assay data for learning QC ideas.
- A z-score flag is a review prompt, not proof that a point is invalid.
- Real QC decisions require raw data, lab notes, instrument context, and study design details.
